# BPIC 2017 loader — with role grouping (Song & Van der Aalst 2008)

Sibling of `BPIC_2017_full_loader.ipynb` that adds the Camargo 2019 pre-processing step missing from the original loader: grouping raw resources into **roles** before feeding them into the LSTM.

### Algorithm (Song & Van der Aalst, *Analysis of Socio-Organizational Structures*, 2008)

1. Build a resource-by-activity frequency matrix `P` (how often each resource performs each activity).
2. Normalise each resource row to a probability distribution over activities.
3. Compute the Pearson correlation matrix between rows — a resource-to-resource *task similarity*.
4. Group resources whose pairwise correlation is ≥ `CORRELATION_THRESHOLD` (we use average-linkage agglomerative clustering on distance = `1 − correlation`).
5. Replace the raw `org:resource` column with the discovered `Role` label.

The output pickle is written to `encoded_data/compare_camargo/BPIC_2017_all_5_roles_{train,test,val}.pkl` so the existing raw-resource pickles are left untouched.

Note: the resulting pickle has `categorical_columns = ['concept:name', 'Role']`, so it is **not** drop-in compatible with checkpoints trained on raw `org:resource`. You would retrain the Camargo model on this pickle to close the loader-side gap to the paper's intended pre-processing.

BPIC 2017 has several hundred unique resources; `CORRELATION_THRESHOLD = 0.85` is a starting point, sweep down (0.75–0.85) if you want more aggressive merging.

In [1]:
import importlib
import sys
import torch
import numpy as np
import pandas as pd

sys.path.insert(0, '..')
sys.path.insert(0, '../..')
sys.path.insert(0, '../../..')
sys.path.insert(0, '../../../..')

import event_log_loader.new_event_log_loader
importlib.reload(event_log_loader.new_event_log_loader)
from event_log_loader.new_event_log_loader import EventLogLoader

import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

np.random.seed(17)

In [2]:
# --- Config ---
SRC_CSV = '../../../../data/BPI Challenge 2017.csv'
ROLE_CSV = '../../../../encoded_data/compare_camargo/BPI_Challenge_2017_with_roles.csv'
OUT_DIR = '../../../../encoded_data/compare_camargo/'
RESULT_NAME = 'BPIC_2017_all'
SUFFIX_TAG = 'roles'  # appended so the role-based pickles sit next to the raw-resource ones

RESOURCE_COL = 'org:resource'
ACTIVITY_COL = 'concept:name'
CASE_COL = 'case:concept:name'
ROLE_COL = 'Role'

CORRELATION_THRESHOLD = 0.85  # Song & Van der Aalst recommend ≥ 0.75; 0.85 is a common default

## 1. Discover roles

In [3]:
from sklearn.cluster import AgglomerativeClustering

raw = pd.read_csv(SRC_CSV)
print(f'Event log: {len(raw)} rows | cases: {raw[CASE_COL].nunique()} | unique resources: {raw[RESOURCE_COL].nunique()}')

# 1 + 2: normalised resource × activity profile matrix
profile = raw.pivot_table(index=RESOURCE_COL, columns=ACTIVITY_COL, aggfunc='size', fill_value=0)
row_totals = profile.sum(axis=1).replace(0, 1)
profile_norm = profile.div(row_totals, axis=0)

# 3: Pearson correlation between resources
corr = profile_norm.T.corr()  # [resources, resources]
print(f'Resource profile matrix: {profile.shape}, correlation matrix: {corr.shape}')

# 4: agglomerative clustering with distance_threshold = 1 - correlation_threshold
#    -- average-linkage is equivalent to Song & Van der Aalst's clique-like grouping up to tie-breaking
dist = 1 - corr.clip(-1, 1).values
np.fill_diagonal(dist, 0.0)  # guard against floating noise
cluster = AgglomerativeClustering(
    n_clusters=None,
    distance_threshold=1 - CORRELATION_THRESHOLD,
    metric='precomputed',
    linkage='average',
)
labels = cluster.fit_predict(dist)

# 5: build resource -> role map, renaming clusters as Role_1..Role_K in descending population
resources = corr.index.tolist()
counts = pd.Series(labels, index=resources).value_counts()
rename = {old: f'Role_{rank+1}' for rank, (old, _) in enumerate(counts.items())}
role_map = {r: rename[l] for r, l in zip(resources, labels)}

print(f'Discovered {len(set(role_map.values()))} roles (correlation threshold = {CORRELATION_THRESHOLD}):')
role_to_resources = {}
for r, role in role_map.items():
    role_to_resources.setdefault(role, []).append(r)
for role in sorted(role_to_resources, key=lambda x: -len(role_to_resources[x])):
    members = role_to_resources[role]
    print(f'  {role:<8s} ({len(members):4d} resources): {members[:8]}{" ..." if len(members) > 8 else ""}')

Event log: 1202267 rows | cases: 31509 | unique resources: 149
Resource profile matrix: (149, 26), correlation matrix: (149, 149)
Discovered 16 roles (correlation threshold = 0.85):
  Role_1   (  70 resources): ['User_10', 'User_103', 'User_104', 'User_105', 'User_108', 'User_110', 'User_12', 'User_13'] ...
  Role_2   (  31 resources): ['User_102', 'User_106', 'User_112', 'User_113', 'User_114', 'User_116', 'User_117', 'User_118'] ...
  Role_3   (  13 resources): ['User_11', 'User_14', 'User_141', 'User_23', 'User_26', 'User_28', 'User_3', 'User_39'] ...
  Role_4   (   8 resources): ['User_100', 'User_101', 'User_109', 'User_124', 'User_136', 'User_29', 'User_75', 'User_90']
  Role_5   (   8 resources): ['User_148', 'User_2', 'User_33', 'User_34', 'User_41', 'User_54', 'User_66', 'User_67']
  Role_6   (   3 resources): ['User_107', 'User_115', 'User_129']
  Role_7   (   3 resources): ['User_132', 'User_31', 'User_85']
  Role_8   (   3 resources): ['User_138', 'User_143', 'User_144']
  

## 2. Write role-augmented CSV

EventLogLoader consumes a CSV path, so we persist the augmented table (raw columns + new `Role`) to disk and point the loader at it.

In [4]:
raw[ROLE_COL] = raw[RESOURCE_COL].map(role_map).fillna('Role_unknown')

from pathlib import Path
Path(ROLE_CSV).parent.mkdir(parents=True, exist_ok=True)
raw.to_csv(ROLE_CSV, index=False)
print(f'Wrote {ROLE_CSV} | {len(raw)} rows | role distribution:')
print(raw[ROLE_COL].value_counts().to_string())

Wrote ../../../../encoded_data/compare_camargo/BPI_Challenge_2017_with_roles.csv | 1202267 rows | role distribution:
Role
Role_1     382611
Role_2     338662
Role_11    148404
Role_3     136910
Role_4      90415
Role_5      66562
Role_7      16228
Role_16      9536
Role_6       8713
Role_8       2508
Role_13       876
Role_9        728
Role_12        77
Role_14        23
Role_10        12
Role_15         2


## 3. Run EventLogLoader with `Role` instead of `org:resource`

In [5]:
event_log_properties = {
    'case_name': CASE_COL,
    'concept_name': ACTIVITY_COL,
    'timestamp_name': 'time:timestamp',
    'time_since_case_start_column': 'case_elapsed_time',
    'time_since_last_event_column': 'event_elapsed_time',
    'day_in_week_column': 'day_in_week',
    'seconds_in_day_column': 'seconds_in_day',
    'min_suffix_size': 5,
    'train_validation_size': 0.15,
    'test_validation_size': 0.2,
    'window_size': 'auto',
    'categorical_columns': [ACTIVITY_COL, ROLE_COL],
    'continuous_columns': ['case_elapsed_time'],
    'continuous_positive_columns': [],
}

event_log_loader = EventLogLoader(ROLE_CSV, event_log_properties)
print('window_size:', event_log_loader.encoder_decoder.window_size)

window_size: 96


In [6]:
train_dataset = event_log_loader.get_dataset('train')
path = f"{OUT_DIR}{RESULT_NAME}_{event_log_loader.encoder_decoder.min_suffix_size}_{SUFFIX_TAG}_train.pkl"
torch.save(train_dataset, path)
print(f'Saved {path}')
print(train_dataset.all_categories)

categorical tensors:   0%|          | 0/2 [00:00<?, ?it/s]

concept:name:   0%|          | 0/20482 [00:00<?, ?it/s]

Role:   0%|          | 0/20482 [00:00<?, ?it/s]

continouous tensors:   0%|          | 0/1 [00:00<?, ?it/s]

case_elapsed_time:   0%|          | 0/20482 [00:00<?, ?it/s]

Saved ../../../../encoded_data/compare_camargo/BPIC_2017_all_5_roles_train.pkl
([('concept:name', 28, {'A_Accepted': 1, 'A_Cancelled': 2, 'A_Complete': 3, 'A_Concept': 4, 'A_Create Application': 5, 'A_Denied': 6, 'A_Incomplete': 7, 'A_Pending': 8, 'A_Submitted': 9, 'A_Validating': 10, 'EOS': 11, 'O_Accepted': 12, 'O_Cancelled': 13, 'O_Create Offer': 14, 'O_Created': 15, 'O_Refused': 16, 'O_Returned': 17, 'O_Sent (mail and online)': 18, 'O_Sent (online only)': 19, 'W_Assess potential fraud': 20, 'W_Call after offers': 21, 'W_Call incomplete files': 22, 'W_Complete application': 23, 'W_Handle leads': 24, 'W_Personal Loan collection': 25, 'W_Shortened completion ': 26, 'W_Validate application': 27}), ('Role', 18, {'EOS': 1, 'Role_1': 2, 'Role_10': 3, 'Role_11': 4, 'Role_12': 5, 'Role_13': 6, 'Role_14': 7, 'Role_15': 8, 'Role_16': 9, 'Role_2': 10, 'Role_3': 11, 'Role_4': 12, 'Role_5': 13, 'Role_6': 14, 'Role_7': 15, 'Role_8': 16, 'Role_9': 17})], [('case_elapsed_time', 1, {})])


In [7]:
test_dataset = event_log_loader.get_dataset('test')
path = f"{OUT_DIR}{RESULT_NAME}_{event_log_loader.encoder_decoder.min_suffix_size}_{SUFFIX_TAG}_test.pkl"
torch.save(test_dataset, path)
print(f'Saved {path}')

categorical tensors:   0%|          | 0/2 [00:00<?, ?it/s]

concept:name:   0%|          | 0/6301 [00:00<?, ?it/s]

Role:   0%|          | 0/6301 [00:00<?, ?it/s]

continouous tensors:   0%|          | 0/1 [00:00<?, ?it/s]

case_elapsed_time:   0%|          | 0/6301 [00:00<?, ?it/s]

Saved ../../../../encoded_data/compare_camargo/BPIC_2017_all_5_roles_test.pkl


In [8]:
val_dataset = event_log_loader.get_dataset('val')
path = f"{OUT_DIR}{RESULT_NAME}_{event_log_loader.encoder_decoder.min_suffix_size}_{SUFFIX_TAG}_val.pkl"
torch.save(val_dataset, path)
print(f'Saved {path}')

categorical tensors:   0%|          | 0/2 [00:00<?, ?it/s]

concept:name:   0%|          | 0/4726 [00:00<?, ?it/s]

Role:   0%|          | 0/4726 [00:00<?, ?it/s]

continouous tensors:   0%|          | 0/1 [00:00<?, ?it/s]

case_elapsed_time:   0%|          | 0/4726 [00:00<?, ?it/s]

Saved ../../../../encoded_data/compare_camargo/BPIC_2017_all_5_roles_val.pkl


### Next step

Clone `src/reimplemented_comparable_approaches/camargo_LSTM_suffix_pred/notebooks/training/Helpdesk/train_camargo_LSTM_sharedcat_roles.ipynb` for BPIC 2017, point it at the new `_roles_` pickles produced above, and bump the n-gram size from 5 → 15 (BPIC 2017 has long, complex traces — longer n-grams help per Camargo 2019, Fig. 7b).

Threshold tuning: lower `CORRELATION_THRESHOLD` to merge more aggressively. BPIC 2017 has ≈ hundreds of `User_*` resources, so 0.85 may leave too many singletons; try 0.80 or 0.75 if the role count stays implausibly high.